In [20]:
import anthropic
import json
import numpy as np
import pandas as pd

client = anthropic.Anthropic()
def get_car_specs(make, model, year, fuel_type):
    prompt = f"""For the {year} {make} {model} with fuel type "{fuel_type}", provide the following in strict JSON format only, no other text:
{{
  "cylinders": <number, integer, or null if electric with no engine>,
  "fuel_efficiency_l_per_100km": <number, float, or null if fully electric>,
  "country_of_origin": "<string>"
}}
Consider that the fuel type may indicate a specific trim/variant (e.g. M-performance, Hybrid, Electric) which can have different specs than the base model, and that specs can vary by model year due to generation changes or manufacturing location shifts.
If unsure about a specific value, use null."""

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    )
    
    text = response.content[0].text.strip()
    text = text.replace("```json", "").replace("```", "").strip()
    return json.loads(text)

In [21]:
df= pd.read_csv("../PreEDAVersion.csv")

In [22]:
unique_cars = df[["make_model", "fuel_type"]].drop_duplicates().reset_index(drop=True)
unique_cars

,make_model,fuel_type
0,2022 Lincoln Corsair,Gas
1,2026 Jeep Cherokee,Gas/Electric Hybrid
2,2026 Hyundai Tucson,Gas
3,2024 Mazda CX-5,Gas
4,2021 Ford F-150,Gas
...,...,...
4660,2016 Chevrolet Suburban,Gas
4661,2023 Mercedes-Benz S-Class,Gas
4662,2009 Porsche Cayman 718,Gas
4663,2009 Mercedes-Benz CLK-Class,Gas


In [23]:
def get_car_specs(make_model, fuel_type):
    prompt = f"""For the car "{make_model}" with fuel type "{fuel_type}", provide the following in strict JSON format only, no other text:
{{
  "cylinders": <number, integer, or null if electric with no engine>,
  "fuel_efficiency_l_per_100km": <number, float, or null if fully electric>,
  "country_of_origin": "<string>"
}}
Consider that the fuel type may indicate a specific trim/variant (e.g. M-performance, Hybrid, Electric) which can have different specs than the base model, and that specs can vary by model year due to generation changes or manufacturing location shifts.
If unsure about a specific value, use null."""

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    )
    
    text = response.content[0].text.strip()
    text = text.replace("```json", "").replace("```", "").strip()
    return json.loads(text)

In [ ]:
unique_cars = df[["make_model", "fuel_type"]].drop_duplicates().reset_index(drop=True)

results = []

for idx, row in unique_cars.iterrows():
    try:
        specs = get_car_specs(row["make_model"], row["fuel_type"])
        specs["make_model"] = row["make_model"]
        specs["fuel_type"] = row["fuel_type"]
        print(f"Success for {row['make_model']} ({row['fuel_type']})")
        results.append(specs)
    except Exception as e:
        print(f"Failed for {row['make_model']} ({row['fuel_type']}): {e}")
        results.append({"make_model": row["make_model"], "fuel_type": row["fuel_type"],
                        "cylinders": None, "fuel_efficiency_l_per_100km": None,
                        "country_of_origin": None})

specs_df = pd.DataFrame(results).to_csv("New Spec")
#df = df.merge(specs_df, on=["make_model", "fuel_type"], how="left")

Success for 2022 Lincoln Corsair (Gas)
Success for 2026 Jeep Cherokee (Gas/Electric Hybrid)
Success for 2026 Hyundai Tucson (Gas)
Success for 2024 Mazda CX-5 (Gas)
Success for 2021 Ford F-150 (Gas)
Success for 2024 Land Rover Defender (Gas)
Success for 2022 Honda Passport (Gas)
Success for 2023 Kia Seltos (Gas)
Success for 2023 Genesis GV70 (Gas)
Success for 2022 Audi S5 Sportback (Gas)
Success for 2019 Mazda Mazda3 (Gas)
Success for 2020 Subaru Outback (Gas)
Success for 2020 Land Rover Defender (Gas/Electric Hybrid)
Success for 2025 Toyota Prius (Gas/Electric Hybrid)
Success for 2022 BMW X5 (Electric)
Success for 2025 Subaru WRX (Gas)
Success for 2020 Infiniti QX60 (Gas)
Success for 2021 Ford Bronco Sport (Gas)
Success for 2017 Honda Civic Sedan (Gas)
Success for 2023 Nissan Versa (Gas)
Success for 2015 Mercedes-Benz GLA (Gas)
Success for 2011 Land Rover Range Rover Sport (Gas)
Success for 2024 Volkswagen Taos (Gas)
Success for 2022 Volkswagen Tiguan (Gas)
Success for 2025 Kia Carniva

TypeError: Can only merge Series or DataFrame objects, a <class 'NoneType'> was passed

In [ ]:
def get_body_type(make, model, year, more_info):
    info_line = f'Additional info: "{more_info}"' if pd.notna(more_info) else "No additional info provided."
    
    prompt = f"""For the {year} {make} {model}, determine its body type (e.g. Sedan, SUV, Coupe, Hatchback, Pickup Truck, Convertible, Wagon, Minivan).

{info_line}

Consider that this exact make, model, and year combination may have a different body type than other years of the same nameplate, since manufacturers sometimes change body style across generations.

If the additional info explicitly states or clearly implies the body type, use that. Otherwise, use your own knowledge of this specific make, model, and year to determine the most likely body type.

Respond in strict JSON format only, no other text:
{{
  "body_type": "<string>"
}}"""

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=100,
        messages=[{"role": "user", "content": prompt}]
    )
    
    text = response.content[0].text.strip()
    text = text.replace("```json", "").replace("```", "").strip()
    return json.loads(text)

In [ ]:
unique_cars = df[["Year","Make", "Model", "MoreInformation"]].drop_duplicates().reset_index(drop=True)
unique_cars

,Year,Make,Model,MoreInformation
0,2022.0,Lincoln,Corsair,NaN
1,2026.0,Jeep,Cherokee,NaN
2,2026.0,Hyundai,Tucson,NaN
3,2024.0,Mazda,CX-5,NaN
4,2021.0,Ford,F-150,NaN
...,...,...,...,...
4252,2020.0,Nissan,Pathfinder,NaN
4253,2016.0,Chevrolet,Suburban,NaN
4254,2009.0,Porsche,Cayman,718
4255,2009.0,Mercedes-Benz,CLK-Class,NaN


In [ ]:
unique_cars = df[["Make", "Model", "MoreInformation"]].drop_duplicates().reset_index(drop=True)

results = []

for idx, row in unique_cars.iterrows():
    try:
        specs = get_body_type(row["Make"], row["Model"], row["MoreInformation"])
        specs["Make"] = row["Make"]
        specs["Model"] = row["Model"]
        specs["MoreInformation"] = row["MoreInformation"]
        print(f"Success for {row['Make']} {row['Model']}")
        results.append(specs)
    except Exception as e:
        print(f"Failed for {row['Make']} {row['Model']}: {e}")
        results.append({"Make": row["Make"], "Model": row["Model"], 
                        "MoreInformation": row["MoreInformation"], "body_type": None})

body_type_df = pd.DataFrame(results)

Success for Lincoln Corsair
Succceful for Lincoln Corsair 
Success for Jeep Cherokee
Succceful for Jeep Cherokee 
Success for Hyundai Tucson
Succceful for Hyundai Tucson 
Success for Mazda CX-5
Succceful for Mazda CX-5 
Success for Ford F-150
Succceful for Ford F-150 
Success for Land Rover
Succceful for Land Rover 
Success for Honda Passport
Succceful for Honda Passport 
Success for Kia Seltos
Succceful for Kia Seltos 
Success for Genesis GV70
Succceful for Genesis GV70 
Success for Audi S5
Succceful for Audi S5 
Success for Mazda Mazda3
Succceful for Mazda Mazda3 
Success for Subaru Outback
Succceful for Subaru Outback 
Success for Toyota Prius
Succceful for Toyota Prius 
Success for BMW X5
Succceful for BMW X5 
Success for Subaru WRX
Succceful for Subaru WRX 
Success for Infiniti QX60
Succceful for Infiniti QX60 
Success for Ford Bronco
Succceful for Ford Bronco 
Success for Honda Civic
Succceful for Honda Civic 
Success for Nissan Versa
Succceful for Nissan Versa 
Success for Merce

In [ ]:
body_type_df.to_csv("BodyType.csv")